# GPT-6.1 Terra — ZERO-SHOT Experiment A (reference only) — FINAL RUN

Closed-model replication of the open-source experiments. Prompt wording is identical to the Qwen / LLaVA / Gemma notebooks of the same condition.

* Model: `gpt-6.1-terra` via the **Responses API**, reasoning effort **medium**, images resized with `thumbnail((512, 512))` exactly like the open-source runs.
* The pilot is a separate notebook (`GPT_ExperimentA_ZeroShot_Pilot.ipynb`); this notebook is the full 1600-image run.
* Full run = **Batch API** (50% cheaper). Batch ids are saved to Drive, so a Colab disconnect loses nothing.
* Add `OPENAI_API_KEY` in Colab **Secrets** (key icon). Never paste it into a cell.

## 1. Mount Drive & confirm input data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from zipfile import ZipFile

zip_path = "/content/drive/MyDrive/CrowdDensityResearch/Phase5_Input.zip"

if not os.path.exists("/content/Phase5_Input"):
    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("/content/")

print("Input ready:", os.path.exists("/content/Phase5_Input"))

Input ready: True


## 2. Back up any existing GPT Experiment A results before overwriting

In [3]:
import os, shutil

RESULT_DIR = "/content/drive/MyDrive/CrowdDensityResearch/VLM_Results"
os.makedirs(RESULT_DIR, exist_ok=True)

for fname in ["gpt_experiment_A_zeroshot.csv", "gpt_experiment_A_zeroshot_evaluated.csv"]:
    src = os.path.join(RESULT_DIR, fname)
    if os.path.exists(src):
        backup = os.path.join(RESULT_DIR, fname.replace(".csv", "_backup.csv"))
        if not os.path.exists(backup):
            shutil.copy2(src, backup)
            print(f"Backed up {fname} -> {os.path.basename(backup)}")
        else:
            print(f"Backup already exists for {fname}, skipping.")
    else:
        print(f"No existing {fname} found — nothing to back up.")

No existing gpt_experiment_A_zeroshot.csv found — nothing to back up.
No existing gpt_experiment_A_zeroshot_evaluated.csv found — nothing to back up.


## 3. Install packages

In [4]:
!pip install -q -U openai

## 4. Connect to GPT-6.1 Terra & configuration
Everything that affects the results is set here. `temperature` / `top_p` are **not** sent (not allowed while reasoning is on), so runs are not strictly deterministic.
Prices are third-party-reported — check them on OpenAI's pricing page. `QUEUE_TOKEN_LIMIT` must match **your tier's** batch queue limit (your org's limit is 1,350,000, as reported by the API).

In [5]:
import os, io, re, json, time, math, base64, random, hashlib
from functools import lru_cache
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

COND = "zs_A"
MODEL            = "gpt-6.1-terra"
REASONING_EFFORT = "medium"      # none | low | medium | high | xhigh | max
IMAGE_DETAIL     = "high"        # low | high | original | auto
MAX_OUTPUT_TOKENS = 25000        # cap incl. reasoning tokens (not a cost)

PRICE_IN, PRICE_OUT = 1.00, 5.00 # USD per 1M tokens — VERIFY
BATCH_DISCOUNT = 0.5

MAX_FILE_MB       = 100          # hard limit is 200 MB per batch file
QUEUE_TOKEN_LIMIT = 1_350_000    # your org's enqueued-token limit for gpt-6.1-terra (from the API error message)
QUEUE_SAFETY      = 0.85

CONFIG = dict(model=MODEL, effort=REASONING_EFFORT, detail=IMAGE_DETAIL, max_out=MAX_OUTPUT_TOKENS, image_max=512, cond=COND)
CONFIG_HASH = hashlib.sha1(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:10]
print(CONFIG, CONFIG_HASH)

{'model': 'gpt-6.1-terra', 'effort': 'medium', 'detail': 'high', 'max_out': 25000, 'image_max': 512, 'cond': 'zs_A'} e30084fdbc


## 5. Helpers

In [6]:
from PIL import Image

def load_image(path, max_size=512):
    """Same resize as the open-source runs."""
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_size, max_size))
    return img

DENSITY_TEXT = {
    0: "Low",
    1: "Medium",
    2: "High"
}

@lru_cache(maxsize=3000)
def img_data_url(path):
    buf = io.BytesIO()
    load_image(path).save(buf, format="JPEG", quality=95)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

def _img(path):
    return {"type": "input_image", "image_url": img_data_url(path), "detail": IMAGE_DETAIL}

## 6. Zero-shot prompt builder — Experiment A
System prompt is the zero-shot version of the open-source notebooks. The builder returns one Responses-API request body (used for both the pilot and the Batch file).

In [7]:
SYSTEM_PROMPT = """You are an expert in crowd counting and crowd density estimation.

You will receive one image.

Predict:

1. The EXACT crowd count.
2. The crowd density.

Density labels:
0 = Low
1 = Medium
2 = High

Return ONLY:

Count: <integer>
Density: <0/1/2>

Do not explain.
Do not estimate.
Do not output ranges.
Return exactly one integer count.
"""

TEST_TURN = """TEST IMAGE

Count every visible person exactly once.

Return ONLY

Count: <integer>

Density: <0/1/2>
"""


def build_messages_zeroshot_A(test_image):
    """Responses-API request for one test image — zero-shot Experiment A."""
    inp = [{"role": "user", "content": [_img(test_image), {"type": "input_text", "text": TEST_TURN}]}]
    return {"model": MODEL, "instructions": SYSTEM_PROMPT, "input": inp,
            "reasoning": {"effort": REASONING_EFFORT}, "max_output_tokens": MAX_OUTPUT_TOKENS, "store": False}

## 7. Inference and parsing (same robust parser used throughout this project)

In [8]:
def run_gpt(request, retries=5):
    """One synchronous call (pilot / smoke test only). Returns (response_dict, error_string)."""
    last = None
    for attempt in range(retries):
        try:
            return client.responses.create(**request).model_dump(), None
        except Exception as e:
            last = f"{type(e).__name__}: {e}"
            time.sleep(2 ** attempt)
    return None, last


def parse_response(response):
    """Rejects degenerate long digit-runs as unparseable rather than taking them literally."""
    response = str(response)

    count = None
    m = re.search(r"Count\s*[:=]\s*(\d+)", response, re.IGNORECASE)
    if m:
        digits = m.group(1)
        if len(digits) <= 5:
            count = int(digits)

    density = None
    m = re.search(r"Density\s*[:=]\s*([012])", response, re.IGNORECASE)
    if m:
        density = int(m.group(1))
    else:
        r = response.lower()
        if "low" in r:
            density = 0
        elif "medium" in r:
            density = 1
        elif "high" in r:
            density = 2

    return count, density


def extract_text(body):
    if not body:
        return ""
    parts = []
    for item in body.get("output") or []:
        if item.get("type") == "message":
            for c in item.get("content") or []:
                if c.get("type") == "output_text":
                    parts.append(c.get("text", ""))
    return "".join(parts).strip()


def usage_of(body):
    u = (body or {}).get("usage") or {}
    return dict(in_tok=u.get("input_tokens", 0), out_tok=u.get("output_tokens", 0),
                reason_tok=(u.get("output_tokens_details") or {}).get("reasoning_tokens", 0))


# offline self-test (no API call)
_fake = {"status": "completed", "output": [{"type": "reasoning", "summary": []},
         {"type": "message", "content": [{"type": "output_text", "text": "Count: 123\n\nDensity: 2"}]}],
         "usage": {"input_tokens": 900, "output_tokens": 400, "output_tokens_details": {"reasoning_tokens": 380}}}
assert parse_response(extract_text(_fake)) == (123, 2)
assert parse_response("Count: 12345678\nDensity: 1") == (None, 1)
assert usage_of(_fake) == dict(in_tok=900, out_tok=400, reason_tok=380)
print("Parser self-test passed.")

Parser self-test passed.


## 8. Load data & checks (no API calls)
No `support_mapping.csv` needed for zero-shot. Prints the rendered prompt structure so you can confirm it before spending anything.

In [9]:
import pandas as pd
import os

ROOT = "/content/Phase5_Input"
TEST_DIR = f"{ROOT}/test/images"

test_df = pd.read_csv(f"{ROOT}/test.csv")

TEST_FILES = sorted(test_df["filename"].astype(int).tolist())

def test_path(f): return os.path.join(TEST_DIR, f"{int(f):04d}.jpg")

def make_request(f):
    f = int(f)
    return build_messages_zeroshot_A(test_path(f))

# ---- data checks ----
assert len(TEST_FILES) == len(set(TEST_FILES)) == 1600, len(TEST_FILES)
_missing = [test_path(f) for f in TEST_FILES if not os.path.exists(test_path(f))]
assert not _missing, f"{len(_missing)} image files missing, e.g. {_missing[:3]}"
print("Data checks passed:", len(TEST_FILES), "test images")

Data checks passed: 1600 test images


In [10]:
def render(req):
    out = ["[instructions] " + req["instructions"].split("\n")[0] + " ..."]
    for m in req["input"]:
        if isinstance(m["content"], str):
            out.append(f"[{m['role']}] {m['content']!r}")
        else:
            parts = ["<image>" if c["type"] == "input_image" else c["text"] for c in m["content"]]
            out.append(f"[{m['role']}] " + " | ".join(repr(p) for p in parts))
    return "\n".join(out)

_req = make_request(TEST_FILES[0])
_n_img = sum(1 for m in _req["input"] if not isinstance(m["content"], str) for c in m["content"] if c["type"] == "input_image")
assert _n_img == 1, _n_img
assert "temperature" not in _req and _req["reasoning"] == {"effort": REASONING_EFFORT}
assert ("YOLO predicted count: " in render(_req)) == False
print(render(_req))
print("\nImages in request:", _n_img, "| request size:", round(len(json.dumps(_req)) / 1024), "KB")

[instructions] You are an expert in crowd counting and crowd density estimation. ...
[user] '<image>' | 'TEST IMAGE\n\nCount every visible person exactly once.\n\nReturn ONLY\n\nCount: <integer>\n\nDensity: <0/1/2>\n'

Images in request: 1 | request size: 73 KB


In [11]:
def row_from_body(fname, body, err=None):
    """One result row from a Responses-API body (used when assembling the batch results)."""
    text = extract_text(body)
    pc, pd_ = parse_response(text)
    row = dict(test_filename=int(fname), predicted_count=pc, predicted_density=pd_, raw_response=text,
               status=(body or {}).get("status"),
               incomplete=((body or {}).get("incomplete_details") or {}).get("reason"),
               error=err, **usage_of(body))
    pass
    return row

## 9. Full run — all 1600 test images via the Batch API, saved to Drive
**Spends real money** (about $3 for a zero-shot condition at Batch prices, from the pilot; 5-shot sends 6 images per request and costs somewhat more). Two steps:
* **9a** builds the batch file(s) (split under 100 MB each) and submits them. Gated by `CONFIRM_FULL_RUN`. Re-run it any time: it only submits what is still pending and stays under your queue limit.
* **9b** (run in this or any later session) checks status, downloads the results and writes `gpt_experiment_A_zeroshot.csv` to Drive. Batch ids live in a state file on Drive.

In [13]:
CONFIRM_FULL_RUN = False         # <-- set True only after the pilot verdict is GO

STATE_PATH = f"{RESULT_DIR}/gpt_batch_state_{COND}_{CONFIG_HASH}.json"
BATCH_LOCAL = f"/content/gpt_batch_{COND}"
RAW_DIR = f"{RESULT_DIR}/gpt_raw_batch_{COND}_{CONFIG_HASH}"
os.makedirs(BATCH_LOCAL, exist_ok=True); os.makedirs(RAW_DIR, exist_ok=True)
TERMINAL = {"completed", "failed", "expired", "cancelled"}
IN_FLIGHT = {"validating", "in_progress", "finalizing", "cancelling"}

def load_state():
    return json.load(open(STATE_PATH)) if os.path.exists(STATE_PATH) else None

def save_state(st):
    json.dump(st, open(STATE_PATH + ".tmp", "w"), indent=1); os.replace(STATE_PATH + ".tmp", STATE_PATH)

def est_tokens(n):
    per = 858
    return int(per * 1.1 * n)

def build_batch_files():
    old = load_state(); old_by = {e["file"]: e for e in old["entries"]} if old else {}
    entries, part, lines, size = [], 0, [], 0
    def flush():
        nonlocal part, lines, size
        if not lines: return
        part += 1; name = f"{COND}_part{part:03d}.jsonl"; path = f"{BATCH_LOCAL}/{name}"
        open(path, "w", encoding="utf-8").write("".join(lines))
        entries.append(dict(file=name, path=path, n=len(lines), bytes=size, est_tokens=est_tokens(len(lines)),
                            input_file_id=None, batch_id=None, status=None, output_file_id=None, error_file_id=None, downloaded=False))
        lines, size = [], 0
    for f in TEST_FILES:
        line = json.dumps({"custom_id": f"{COND}__{int(f):04d}", "method": "POST", "url": "/v1/responses", "body": make_request(f)}) + "\n"
        if lines and size + len(line.encode()) > MAX_FILE_MB * 1024 * 1024: flush()
        lines.append(line); size += len(line.encode())
    flush()
    for e in entries:                       # keep submission progress of identical files
        o = old_by.get(e["file"])
        if o and o["n"] == e["n"]:
            for k in ("input_file_id", "batch_id", "status", "output_file_id", "error_file_id", "downloaded"): e[k] = o[k]
    st = dict(config=CONFIG, entries=entries); save_state(st); return st

def refresh_states(st):
    for e in st["entries"]:
        if e["batch_id"] and (e["status"] not in TERMINAL or (e["status"] == "completed" and not e["output_file_id"] and not e["downloaded"])):
            b = client.batches.retrieve(e["batch_id"])
            e["status"], e["output_file_id"], e["error_file_id"] = b.status, b.output_file_id, b.error_file_id
            e["counts"] = b.request_counts.model_dump() if b.request_counts else None
    save_state(st)

st = load_state()
if st is None or not all(os.path.exists(e["path"]) for e in st["entries"] if not e["batch_id"]):
    t0 = time.time(); st = build_batch_files(); print(f"Built batch files in {time.time() - t0:.0f}s")
assert sum(e["n"] for e in st["entries"]) == len(TEST_FILES), "request count mismatch"
print("Files:", [(e["file"], e["n"], round(e["bytes"] / 1e6, 1)) for e in st["entries"]])

if CONFIRM_FULL_RUN:
    refresh_states(st)
    for e in st["entries"]:        # resubmit parts that were rejected / expired without any output
        if e["batch_id"] and e["status"] in ("failed", "expired", "cancelled") and not e["output_file_id"]:
            print("resetting", e["file"], "(", e["status"], ") -> will be resubmitted")
            e.update(input_file_id=None, batch_id=None, status=None, output_file_id=None, error_file_id=None, downloaded=False, counts=None)
    save_state(st)
    in_flight = sum(e["est_tokens"] for e in st["entries"] if e["batch_id"] and e["status"] in IN_FLIGHT)
    for e in st["entries"]:
        if e["batch_id"]: continue
        if in_flight + e["est_tokens"] > QUEUE_TOKEN_LIMIT * QUEUE_SAFETY:
            print(f"Queue budget reached ({in_flight:,} est. tokens in flight). Re-run this cell later."); break
        up = client.files.create(file=open(e["path"], "rb"), purpose="batch")
        b = client.batches.create(input_file_id=up.id, endpoint="/v1/responses", completion_window="24h",
                                  metadata={"cond": COND, "file": e["file"], "cfg": CONFIG_HASH})
        e.update(input_file_id=up.id, batch_id=b.id, status=b.status, output_file_id=b.output_file_id, error_file_id=b.error_file_id)
        save_state(st)                      # persist the batch id IMMEDIATELY
        in_flight += e["est_tokens"]
        print("submitted", e["file"], "->", b.id, b.status)
else:
    print("CONFIRM_FULL_RUN is False -> nothing submitted.")

Files: [('zs_A_part001.jsonl', 862, 104.8), ('zs_A_part002.jsonl', 738, 93.1)]
CONFIRM_FULL_RUN is False -> nothing submitted.


In [14]:
# 9b — status, download, assemble  (safe to re-run; works in a new session too)
OUT_PATH = os.path.join(RESULT_DIR, "gpt_experiment_A_zeroshot.csv")
st = load_state(); refresh_states(st)

for e in st["entries"]:
    if e["batch_id"] and e["status"] in TERMINAL and not e["downloaded"]:
        if e["output_file_id"]:
            open(f"{RAW_DIR}/{e['file']}.out.jsonl", "w", encoding="utf-8").write(client.files.content(e["output_file_id"]).text)
        if e["error_file_id"]:
            open(f"{RAW_DIR}/{e['file']}.err.jsonl", "w", encoding="utf-8").write(client.files.content(e["error_file_id"]).text)
        e["downloaded"] = True; save_state(st)

for e in st["entries"]:
    print(e["file"], "|", e["batch_id"], "|", e["status"], "|", e.get("counts"), "| downloaded:", e["downloaded"])

got = {}
for p in sorted(os.listdir(RAW_DIR), key=lambda x: (x.startswith("sync_fixes"), x)):   # sync fixes applied last
    if p.endswith(".out.jsonl"):
        for line in open(f"{RAW_DIR}/{p}", encoding="utf-8"):
            if line.strip():
                d = json.loads(line); got[d["custom_id"]] = d      # matched by custom_id: output order is not guaranteed

rows, missing = [], []
for f in TEST_FILES:
    d = got.get(f"{COND}__{int(f):04d}")
    if d is None: missing.append(f); continue
    resp = d.get("response") or {}
    if resp.get("status_code") == 200:
        rows.append(row_from_body(f, resp["body"]))
    else:
        rows.append(row_from_body(f, None, err=json.dumps(d.get("error") or resp)[:300]))

results = pd.DataFrame(rows)
if len(results):
    results.to_csv(OUT_PATH, index=False)
    batch_cost = (results.in_tok.sum() * PRICE_IN + results.out_tok.sum() * PRICE_OUT) / 1e6 * BATCH_DISCOUNT
    print(f"\nAssembled {len(results)}/{len(TEST_FILES)} rows -> {OUT_PATH}  (actual cost at Batch prices: ${batch_cost:.2f})")
if missing:
    print(f"WAITING / MISSING: {len(missing)} results not available yet, e.g. {missing[:5]}. Re-run this cell when the batches finish.")
failed = int(results.predicted_count.isna().sum()) if len(results) else 0
print("Unparseable counts:", failed)

zs_A_part001.jsonl | batch_6aef9614eb4c81909730cd40b91bb527 | completed | {'completed': 862, 'failed': 0, 'total': 862} | downloaded: True
zs_A_part002.jsonl | batch_6aef96210d44819085f17b4266a1d276 | completed | {'completed': 738, 'failed': 0, 'total': 738} | downloaded: True

Assembled 1600/1600 rows -> /content/drive/MyDrive/CrowdDensityResearch/VLM_Results/gpt_experiment_A_zeroshot.csv  (actual cost at Batch prices: $2.90)
Unparseable counts: 0


### 9c. Optional - re-run missing or errored requests synchronously
Only needed if some batch requests expired or errored. Gated; results are merged by `custom_id` the next time 9b runs.

In [ ]:
RERUN_MISSING = False             # True = re-run missing / errored / incomplete requests synchronously (full price, not Batch)

if RERUN_MISSING:
    bad = set(missing) | (set(results.loc[results.error.notna() | results.incomplete.notna(), "test_filename"]) if len(results) else set())
    print("To re-run:", len(bad))

    def _fix(f):
        body, err = run_gpt(make_request(f))
        return int(f), body, err

    with open(f"{RAW_DIR}/sync_fixes.out.jsonl", "a", encoding="utf-8") as out, ThreadPoolExecutor(4) as ex:
        for f, body, err in ex.map(_fix, sorted(bad)):
            if body:
                out.write(json.dumps({"custom_id": f"{COND}__{f:04d}", "response": {"status_code": 200, "body": body}, "error": None}) + "\n")
    print("Done -> re-run the 9b cell above to re-assemble.")
else:
    print("RERUN_MISSING is False -> nothing re-run.")

## 10. Evaluate — counting metrics

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

assert len(results) == len(TEST_FILES), f"Only {len(results)}/{len(TEST_FILES)} results are in - finish step 9b first."
gpt_A_zs = results.copy()

gpt_A_zs_eval = gpt_A_zs.merge(
    test_df[["filename", "total_count", "crowd_density"]],
    left_on="test_filename",
    right_on="filename",
    how="left"
)

print("Missing ground truth after merge:", gpt_A_zs_eval["total_count"].isna().sum())

n_before = len(gpt_A_zs_eval)
count_valid = gpt_A_zs_eval.dropna(subset=["predicted_count"]).copy()
n_dropped = n_before - len(count_valid)
print(f"Dropped {n_dropped}/{n_before} rows with no parseable count ({100*n_dropped/n_before:.1f}%)")

y_true_count = count_valid["total_count"]
y_pred_count = count_valid["predicted_count"]

mae = mean_absolute_error(y_true_count, y_pred_count)
rmse = np.sqrt(mean_squared_error(y_true_count, y_pred_count))

nonzero = count_valid["total_count"] > 0
mape = np.mean(np.abs(
    (count_valid.loc[nonzero, "total_count"] - count_valid.loc[nonzero, "predicted_count"])
    / count_valid.loc[nonzero, "total_count"]
)) * 100

print("GPT Experiment A (Zero-Shot) — Counting")
print("=" * 44)
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAPE : {mape:.2f}%")
print(f"(computed on {len(count_valid)}/{n_before} images with a parseable count)")

Missing ground truth after merge: 0
Dropped 0/1600 rows with no parseable count (0.0%)
GPT Experiment A (Zero-Shot) — Counting
MAE  : 198.45
RMSE : 773.75
MAPE : 54.86%
(computed on 1600/1600 images with a parseable count)


## 11. Evaluate — density classification

In [16]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

density_valid = gpt_A_zs_eval.dropna(subset=["predicted_density"]).copy()
n_density_dropped = len(gpt_A_zs_eval) - len(density_valid)
print(f"Dropped {n_density_dropped}/{len(gpt_A_zs_eval)} rows with no parseable density ({100*n_density_dropped/len(gpt_A_zs_eval):.1f}%)")

y_true_density = density_valid["crowd_density"]
y_pred_density = density_valid["predicted_density"]

accuracy = accuracy_score(y_true_density, y_pred_density)
precision = precision_score(y_true_density, y_pred_density, average="weighted", zero_division=0)
recall = recall_score(y_true_density, y_pred_density, average="weighted", zero_division=0)
f1 = f1_score(y_true_density, y_pred_density, average="weighted", zero_division=0)

print("GPT Experiment A (Zero-Shot) — Density")
print("=" * 44)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"(trivial 'always High' baseline: {(test_df['crowd_density'] == 2).mean():.4f})")

cm = confusion_matrix(y_true_density, y_pred_density, labels=[0, 1, 2])
print()
print("Confusion matrix (rows=GT, cols=pred):")
print(cm)

Dropped 0/1600 rows with no parseable density (0.0%)
GPT Experiment A (Zero-Shot) — Density
Accuracy : 0.7519
Precision: 0.7350
Recall   : 0.7519
F1-score : 0.7100
(trivial 'always High' baseline: 0.6644)

Confusion matrix (rows=GT, cols=pred):
[[  74  112   36]
 [   2   70  243]
 [   0    4 1059]]


## 12. Save evaluated results to Drive

In [17]:
EVAL_PATH = os.path.join(RESULT_DIR, "gpt_experiment_A_zeroshot_evaluated.csv")
gpt_A_zs_eval.to_csv(EVAL_PATH, index=False)

print("Saved:", len(gpt_A_zs_eval), "rows")
print("Path:", EVAL_PATH)

Saved: 1600 rows
Path: /content/drive/MyDrive/CrowdDensityResearch/VLM_Results/gpt_experiment_A_zeroshot_evaluated.csv
